# Training progress, interactive

Val hidden-pixel MAE per epoch and the physics residual for any selection of runs. Pick categories (`cnn`, `unet`, `pinn/cnn`, `pinn/unet`), then narrow to individual runs; the figure redraws on every change and on **Refresh** (running jobs rewrite `history.json` each epoch). Needs a live Jupyter / VS Code front end (ipywidgets).

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt

plt.ioff()  # figures go only where display() puts them (inside the Output widgets)
import ipywidgets as W
from IPython.display import display, clear_output
from chl_cnn.runs import CATEGORIES, list_runs
from chl_cnn.watch import draw_runs

cat_sel = W.SelectMultiple(
    options=list(CATEGORIES), value=("pinn/t3d",), rows=4, description="category"
)
run_sel = W.SelectMultiple(
    options=[], rows=14, description="runs", layout=W.Layout(width="520px")
)
pattern = W.Text(value="*", description="name glob", layout=W.Layout(width="260px"))
all_runs = W.Checkbox(value=True, description="all runs in the categories")
refresh = W.Button(description="Refresh", icon="refresh")
out = W.Output()


def _runs():
    return list_runs(cat_sel.value, pattern.value or "*")


def _fill_runs(*_):
    rs = _runs()
    keep = set(run_sel.value)
    run_sel.options = [
        (
            f"{p.parent.parent.name + '/' if p.parent.parent.name == 'pinn' else ''}{p.parent.name}/{p.name}",
            str(p),
        )
        for p in rs
    ]
    run_sel.value = tuple(v for _, v in run_sel.options if v in keep)
    _draw()


def _draw(*_):
    with out:
        clear_output(wait=True)
        sel = (
            _runs()
            if all_runs.value or not run_sel.value
            else [Path(v) for v in run_sel.value]
        )
        if not sel:
            print("no runs selected")
            return
        rows, fig = draw_runs(sel, title_suffix=f"  ({time.strftime('%H:%M:%S')})")
        fig.set_size_inches(16, 5.6)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
        for cat, name, n, best, st in rows:
            print(f"{cat:10s} {name:28s} {st:8s} epochs {n:3d}  best val {best:.4f}")


cat_sel.observe(_fill_runs, "value")
pattern.observe(_fill_runs, "value")
run_sel.observe(_draw, "value")
all_runs.observe(_draw, "value")
refresh.on_click(_draw)
display(W.HBox([cat_sel, W.VBox([pattern, all_runs, refresh])]), run_sel, out)
_fill_runs()

SelectMultiple(description='runs', layout=Layout(width='520px'), options=(), rows=14, value=())

Output()

Tips: deselect **all runs in the categories** to plot only the highlighted runs (Ctrl/Cmd-click for several). The name glob applies within the chosen categories, e.g. `pinn_v2_*` or `*lam0p3`. The static version is `python -m chl_cnn.watch --cat pinn/cnn,pinn/unet --every 60 --out models/progress_pinn.png`.

## Output visualizer: same 32 validation days for any finished run

Each finished run stores its predictions on every val/test day (`preds.npz`). Works for every store (IO, Black Sea 2 km and 4 km); the store, crop, split and donor settings come from the run's config.json. The 32 days are the run's val days with saved predictions, evenly spaced over the observed-fraction range, so runs on the same store and `--val-stride` show the same days. Top row: visible input, prediction, truth, hidden-pixel error. Bottom row: input at t−1 and t+1 as the model sees it, SST, and the error histogram. A day outside the model's input window (e.g. t+1 for the nowcast) is left blank. Loading a store takes a few seconds the first time (about 30 s for the 2 km Black Sea).

In [ ]:
from chl_cnn.viz import context_for, sample_days, draw_day

day_run = W.Dropdown(options=[], description="run", layout=W.Layout(width="520px"))
day_idx = W.IntSlider(
    value=0, min=0, max=31, description="day", continuous_update=False
)
day_play = W.Play(value=0, min=0, max=31, interval=2500, description="cycle")
W.jslink((day_play, "value"), (day_idx, "value"))
day_out = W.Output()
_days = (
    {}
)  # run -> its 32 sampled val days (same days for runs on the same store and --val-stride)


def _fill_day_runs(*_):
    rs = [
        p
        for p in list_runs(cat_sel.value, pattern.value or "*")
        if (p / "preds.npz").exists()
    ]
    keep = day_run.value
    day_run.options = [(f"{p.parent.name}/{p.name}", str(p)) for p in rs]
    if keep in [v for _, v in day_run.options]:
        day_run.value = keep
    elif day_run.options:
        day_run.value = day_run.options[-1][1]


def _show_day(*_):
    if not day_run.value:
        return
    with day_out:
        clear_output(wait=True)
        r = day_run.value
        if r not in _days:
            print(
                "loading store context (first time per store: a few seconds, ~30 s for the 2 km Black Sea)"
            )
            _days[r] = sample_days(context_for(r), r, n=32)
        days = _days[r]
        day_idx.max = day_play.max = len(days) - 1
        fig = draw_day(r, days[min(day_idx.value, len(days) - 1)])
        clear_output(wait=True)
        display(fig)
        plt.close(fig)


day_run.observe(_show_day, "value")
day_idx.observe(_show_day, "value")
cat_sel.observe(_fill_day_runs, "value")
pattern.observe(_fill_day_runs, "value")
_fill_day_runs()
display(W.VBox([day_run, W.HBox([day_play, day_idx]), day_out]))